# Evaluation, Entity Resolution, And Production Hardening

This notebook exercises evaluation/ranking, entity resolution, and production-hardening utilities: deterministic benchmark mode, tie policy, RapidFuzz/alias resolution, manifests, quarantine, review queue, and PII redaction.

In [ ]:
from pathlib import Path
import sys, json, tempfile

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = next((p for p in Path.cwd().parents if (p / 'src').exists()), Path.cwd())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

print(f'Repo root: {ROOT}')

In [ ]:
from kg_construction.application.benchmark_runner import TechniqueEvaluationRunner, TechniqueEvaluationConfig

runner = TechniqueEvaluationRunner(config=TechniqueEvaluationConfig())
for name, path, source_type in [
    ('text', ROOT / 'examples/data_to_kg/sample.txt', 'text'),
    ('csv', ROOT / 'examples/data_to_kg/sample.csv', 'csv'),
    ('json', ROOT / 'examples/data_to_kg/sample.json', 'json'),
]:
    report = runner.run(input_path=path, source_type=source_type, techniques='applicable')
    print(name, report.run_metadata.tie_breaking_policy)
    for rank, score in enumerate(report.ranked_scores[:3], start=1):
        print(rank, score.technique_name, round(score.final_score, 3), round(score.quality_score, 3), round(score.runtime_score, 3))
    print()

In [ ]:
from kg_construction.domain.models import ConstructionBatch, EntityCandidate, EvidenceRef, SourceRecord
from kg_construction.infrastructure.resolution import AliasTableResolver, ExactKeyEntityResolver, NormalizedStringResolver, RapidFuzzResolver, RapidFuzzResolverConfig

source = SourceRecord.from_text('Recommendation API recommendation api Rec API Recomendation API Feature Store', source_name='memory://resolution')
evidence = [EvidenceRef.build(source, location={'kind': 'unit'}, text='entity')]
entities = [EntityCandidate.build(entity_type='System', name=name, evidence=evidence, extractor_name='unit', technique='unit') for name in [
    'Recommendation API', 'recommendation api', 'Rec API', 'Recomendation API', 'Feature Store'
]]
batch = ConstructionBatch(source_records=[source], entities=entities)
batch = ExactKeyEntityResolver().resolve(batch)
batch = NormalizedStringResolver().resolve(batch)
batch = RapidFuzzResolver(RapidFuzzResolverConfig(threshold=88, review_threshold=99)).resolve(batch)
batch = AliasTableResolver(aliases={'Recommendation API': ['Rec API']}).resolve(batch)
print([(e.name, e.properties.get('canonical_name'), e.properties.get('resolution_confidence'), e.properties.get('review_required')) for e in batch.entities])
print('Audit rows:', len(batch.metadata.get('entity_resolution_audit', [])))

In [ ]:
from pathlib import Path
from kg_construction.application.workflow import ConstructionWorkflow
from kg_construction.infrastructure.extractors.regex import RegexEntityExtractor
from kg_construction.infrastructure.ontology.validation import EvidenceRequiredValidator
from kg_construction.infrastructure.production import ErrorQuarantine, HumanReviewQueue, PIIRedactionPolicy, RunManifestWriter
from kg_construction.infrastructure.loaders.text_loader import TextLoader

class FailingExtractor:
    name = 'failing_extractor'
    def extract(self, source):
        raise RuntimeError('demo failure')

source = TextLoader().load(ROOT / 'examples/data_to_kg/sample.txt')[0]
with tempfile.TemporaryDirectory() as tmpdir:
    workflow = ConstructionWorkflow(
        extractors=[FailingExtractor(), RegexEntityExtractor()],
        validators=[EvidenceRequiredValidator(policy='quarantine')],
        manifest_writer=RunManifestWriter(Path(tmpdir) / 'manifests'),
        quarantine=ErrorQuarantine(Path(tmpdir) / 'quarantine.jsonl'),
        human_review_queue=HumanReviewQueue(Path(tmpdir) / 'review.json'),
        pii_policy=PIIRedactionPolicy(),
        continue_on_error=True,
        pipeline_name='notebook_production_demo',
        config_payload={'notebook': '05'},
    )
    result = workflow.run([source])
    print({
        'manifest_status': result.run_manifest.status if result.run_manifest else None,
        'entities': len(result.batch.entities),
        'quarantine_records': len(result.quarantine_records),
        'review_items': len(result.human_review_items),
        'email_redacted': 'ops@example.com' not in (result.batch.source_records[0].content or ''),
    })

In [ ]:
import io
from kg_construction.infrastructure.production import (
    IncrementalSyncPolicy,
    StructuredLoggingConfig,
    WatermarkStore,
    configure_structured_logging,
)

with tempfile.TemporaryDirectory() as tmpdir:
    stream = io.StringIO()
    logger = configure_structured_logging(
        StructuredLoggingConfig(enabled=True, redact_pii=True),
        stream=stream,
        pii_policy=PIIRedactionPolicy(),
    ).bind(run_id='notebook-05')
    logger.info('source_loaded', source_name='memory://sync', contact='ops@example.com')

    store = WatermarkStore(Path(tmpdir) / 'watermarks.json')
    policy = IncrementalSyncPolicy()
    old_source = SourceRecord.from_text('old', source_name='memory://sync', metadata={'updated_at': '2026-07-09T10:00:00Z'})
    new_source = SourceRecord.from_text('new', source_name='memory://sync', metadata={'updated_at': '2026-07-09T11:00:00Z'})
    first = policy.apply([old_source], store)
    policy.commit(first.eligible_sources, store)
    second = policy.apply([old_source], store)
    third = policy.apply([new_source], store)

print({
    'structured_log_event': json.loads(stream.getvalue())['event'],
    'structured_log_redacted': 'ops@example.com' not in stream.getvalue(),
    'first_eligible': len(first.eligible_sources),
    'old_skipped_after_commit': len(second.skipped_sources),
    'new_eligible_after_watermark': len(third.eligible_sources),
})
